# VIMS spectra at ISS luminous-band positions

For every ISS image that has a manually picked band profile (`post_pipeline/13_profile_slices/picks.json`),
this notebook:
1. Reads the pixel coordinates of the I_max pick (band peak) in the ISS image.
2. Converts those ISS pixel coordinates to RA/Dec in J2000 via SPICE.
3. Finds the VIMS cube acquired at the same time and the VIMS pixel nearest to that sky position.
4. Plots the full VIMS I/F spectrum for that pixel — directly comparable to `cube[S, L].spectrum`
   from the pyvims notebook.

Each row of the `14_contrast_table.csv` corresponds to one such ISS image.

In [ ]:
import re
import math
import json
import os
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import spiceypy as spice
from pyvims import VIMS
from pyvims.wvlns import bad_ir_pixels

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────────
_STORAGE    = Path("/mnt/storage/projects/Thesis/LuminousBandsInERing")
PICKS_DIR   = _STORAGE / "post_pipeline" / "13_profile_slices"
CONTRAST_CSV= _STORAGE / "post_pipeline" / "14_contrast_table.csv"
CUBES_ROOT  = _STORAGE / "old" / "outputs" / "VIMS" / "cubes"
VIMS_CSV    = _STORAGE / "old" / "outputs" / "VIMS" / "enceladus_flybys_with_cube_ids.csv"
OBS_SEQ_CSV = _STORAGE / "pipeline_output" / "obs_sequences.csv"

# ── SPICE ──────────────────────────────────────────────────────────────────────
KERNEL_ARCHIVE = _STORAGE / "cosp_1000_070930_120502"
KERNELS_DIR    = KERNEL_ARCHIVE / "cosp_1000" / "data"
METAKERNELS    = sorted(KERNEL_ARCHIVE.glob("cas_*_070930_120502.tm"))
CASSINI_ID, NAC_ID = -82, -82360

# ── VIMS cache ─────────────────────────────────────────────────────────────────
CUBES_ROOT.mkdir(parents=True, exist_ok=True)
DOWNLOAD = True   # download missing cubes from the VIMS portal

## Load SPICE kernels

In [ ]:
rel_paths = set()
for mk in METAKERNELS:
    text = mk.read_text()
    m = re.search(r"\\begindata(.*?)(?:\\begintext|$)", text, re.DOTALL)
    if m:
        rel_paths.update(re.findall(r"'\$KERNELS/([^']+)'", m.group(1)))

spice.kclear()
loaded = 0
for rel in sorted(rel_paths):
    full = KERNELS_DIR / rel
    if full.exists():
        try:
            spice.furnsh(str(full))
            loaded += 1
        except Exception:
            pass

# Cache pixel scale (same for all ISS NAC images)
_, _, _, _, _bounds = spice.getfov(NAC_ID, 10)
ISS_PLATE = math.atan(abs(_bounds[0][0]) / abs(_bounds[0][2])) / 512.0

print(f"Loaded {loaded}/{len(rel_paths)} SPICE kernels")
print(f"ISS NAC pixel scale: {ISS_PLATE * 1e6:.3f} μrad/px")

## Core helper functions

In [ ]:
def iss_pixel_to_radec(col: float, row: float, et: float) -> tuple[float, float]:
    """Convert ISS NAC pixel (col, row) to RA/Dec (degrees, J2000).

    Inverts the SPICE-derived projection in pipeline/08_detect_bands.py:
      col = 512 - atan2(pos_x, pos_z) / plate
      row = 512 - atan2(pos_y, pos_z) / plate
    """
    angle_x = (512.0 - col) * ISS_PLATE
    angle_y = (512.0 - row) * ISS_PLATE
    v_nac   = np.array([math.tan(angle_x), math.tan(angle_y), 1.0])
    v_nac  /= np.linalg.norm(v_nac)
    mat     = spice.pxform("CASSINI_ISS_NAC", "J2000", et)
    v_j2000 = mat @ v_nac
    ra  = math.degrees(math.atan2(v_j2000[1], v_j2000[0])) % 360.0
    dec = math.degrees(math.asin(float(np.clip(v_j2000[2], -1.0, 1.0))))
    return ra, dec


def find_nearest_vims_pixel(cube, target_ra: float, target_dec: float
                            ) -> tuple[int, int, float]:
    """Return (line, sample, angular_separation_deg) of nearest VIMS pixel.

    cube.rsky has shape (2, NL, NS, 4); axis-1 = line, axis-2 = sample.
    Indexing in pyvims: cube[S+1, L+1] (1-indexed, sample first).
    """
    ra_map  = cube.rsky[0, :, :, :].mean(axis=2)   # (NL, NS)
    dec_map = cube.rsky[1, :, :, :].mean(axis=2)
    cos_dec = math.cos(math.radians(target_dec))
    dra     = (ra_map  - target_ra)  * cos_dec
    ddec    = dec_map - target_dec
    dist    = np.sqrt(dra**2 + ddec**2)
    l, s    = np.unravel_index(dist.argmin(), dist.shape)
    return int(l), int(s), float(dist[l, s])


def load_picks(opusid: str) -> dict | None:
    """Load picks.json for opusid. Returns None if missing or no picks."""
    p = PICKS_DIR / opusid / "picks.json"
    if not p.exists():
        return None
    data = json.loads(p.read_text())
    if not (data.get("picks", {}).get("max") and data.get("picks", {}).get("min")):
        return None
    return data


def imax_pixel(picks: dict) -> tuple[float, float]:
    """Return (x_px, y_px) of the I_max pick (highest t_px among max picks)."""
    maxp = picks["picks"]["max"]
    best = max(maxp, key=lambda p: p["t_px"])
    return float(best["x_px"]), float(best["y_px"])

## Build VIMS cube index

Map each ISS opusid → VIMS flyby code → cube IDs and cube start/stop times.
Cubes are loaded on demand and cached.

In [ ]:
obs_df    = pd.read_csv(OBS_SEQ_CSV)     # image_id → obs_sequence (VIMS flyby code)
flybys_df = pd.read_csv(VIMS_CSV)        # Flyby → Cube_IDs, Date

# Build: obs_sequence → list of cube_ids
flyby_to_cubeids: dict[str, list[str]] = {}
for _, row in flybys_df.iterrows():
    raw = row.get("Cube_IDs", "")
    if isinstance(raw, str) and raw.strip():
        flyby_to_cubeids[row["Flyby"]] = [c.strip() for c in raw.split(",") if c.strip()]

# Build: image_id → obs_sequence  (science filter images only)
imgid_to_flyby = dict(zip(obs_df["image_id"], obs_df["obs_sequence"]))

# Build: flyby → approximate ET midpoint (from Date column)
# Used as fallback when the image_id is not in obs_sequences (CLEAR filter images)
flyby_et_mid: dict[str, float] = {}
for _, row in flybys_df.iterrows():
    date_str = str(row.get("Date", ""))
    try:
        et = spice.str2et(date_str.replace("-", " ") + " 00:00:00 UTC")
        flyby_et_mid[row["Flyby"]] = et
    except Exception:
        pass

# Cube cache: cube_id → VIMS object (loaded lazily)
_cube_cache: dict[str, VIMS | None] = {}

def get_cube(cube_id: str) -> VIMS | None:
    """Load a VIMS cube, caching it.  Returns None if unavailable."""
    if cube_id not in _cube_cache:
        try:
            c = VIMS(cube_id, root=str(CUBES_ROOT), download=DOWNLOAD)
            _ = c.data
            _cube_cache[cube_id] = c
        except Exception:
            _cube_cache[cube_id] = None
    return _cube_cache[cube_id]


def find_covering_cube(image_id: str, target_et: float, tol_s: float = 120.0
                       ) -> VIMS | None:
    """Return the VIMS cube whose time range covers target_et.

    Tries the obs_sequences flyby first, then falls back to the nearest
    flyby by date (for CLEAR filter images not in obs_sequences).
    """
    flyby = imgid_to_flyby.get(image_id)

    # Fallback: find the flyby whose date ET is closest to target_et
    if flyby is None and flyby_et_mid:
        flyby = min(flyby_et_mid, key=lambda f: abs(flyby_et_mid[f] - target_et))

    cube_ids = flyby_to_cubeids.get(flyby, [])

    best_cube, best_dt = None, tol_s
    for cid in cube_ids:
        c = get_cube(cid)
        if c is None:
            continue
        et_min = c.et.min()
        et_max = c.et.max()
        if et_min <= target_et <= et_max:
            return c   # exact temporal overlap
        dt = min(abs(target_et - et_min), abs(target_et - et_max))
        if dt < best_dt:
            best_dt, best_cube = dt, c
    return best_cube


print(f"ISS image → flyby map: {len(imgid_to_flyby)} entries")
print(f"VIMS flyby → cube IDs: {len(flyby_to_cubeids)} flybys")
print(f"Flyby ET index (fallback): {len(flyby_et_mid)} flybys")

## Resolve each ISS band pick to a VIMS spectrum

In [ ]:
import pickle as _pkl

_RESOLVE_CACHE = CONTRAST_CSV.parent / "17_resolve_cache.pkl"
_FORCE_RECOMPUTE = False   # set True to ignore the cache and recompute

contrast_df = pd.read_csv(CONTRAST_CSV)   # always needed by downstream cells

if not _FORCE_RECOMPUTE and _RESOLVE_CACHE.exists():
    with open(_RESOLVE_CACHE, "rb") as _f:
        records = _pkl.load(_f)
    matched = [r for r in records if r["spectrum"] is not None]
    print(f"Loaded {len(records)} records from cache ({len(matched)} matched).")
    print(f"  → delete {_RESOLVE_CACHE.name} or set _FORCE_RECOMPUTE=True to regenerate.")
else:
    print(f"Processing {len(contrast_df)} images from 14_contrast_table.csv")

    records = []   # one per ISS image

    for _, row in contrast_df.iterrows():
        opusid   = row["opusid"]
        image_id = "N" + opusid.split("-n")[-1].upper()

        picks = load_picks(opusid)
        if picks is None:
            print(f"  {opusid}: no picks — skipped")
            continue

        # I_max pixel coordinates in the ISS image
        col_px, row_px = imax_pixel(picks)

        # SPICE: ET of the ISS image
        try:
            et = spice.scs2e(CASSINI_ID, "1/" + image_id[1:])
        except Exception as e:
            print(f"  {opusid}: SPICE ET failed ({e}) — skipped")
            continue

        # ISS pixel → RA/Dec
        try:
            ra, dec = iss_pixel_to_radec(col_px, row_px, et)
        except Exception as e:
            print(f"  {opusid}: RA/Dec failed ({e}) — skipped")
            continue

        # Find covering VIMS cube
        cube = find_covering_cube(image_id, et)
        if cube is None:
            print(f"  {opusid}: no VIMS cube found")
            records.append({"opusid": opusid, "image_id": image_id,
                            "band_ra": ra, "band_dec": dec,
                            "cube_id": None, "vims_l": None, "vims_s": None,
                            "sep_deg": None, "spectrum": None, "wvlns": None,
                            "delta": row["delta"], "I_max": row["I_max"]})
            continue

        # Nearest VIMS pixel
        l, s, sep_deg = find_nearest_vims_pixel(cube, ra, dec)

        # Extract spectrum (pyvims: cube[S+1, L+1], 1-indexed)
        pix      = cube[s + 1, l + 1]
        spectrum = pix.spectrum.copy()
        wvlns    = cube.wvlns.copy()

        records.append({
            "opusid":   opusid,
            "image_id": image_id,
            "band_ra":  ra,
            "band_dec": dec,
            "cube_id":  str(cube),
            "vims_l":   l,
            "vims_s":   s,
            "sep_deg":  sep_deg,
            "spectrum": spectrum,
            "wvlns":    wvlns,
            "delta":    row["delta"],
            "I_max":    row["I_max"],
        })

        print(f"  {opusid}: band @ ({col_px:.0f},{row_px:.0f})px → "
              f"RA={ra:.3f}° Dec={dec:.3f}°  "
              f"VIMS l={l} s={s} sep={sep_deg:.3f}°  "
              f"Δ={row['delta']:.3f}")

    matched = [r for r in records if r["spectrum"] is not None]
    print(f"\nMatched {len(matched)}/{len(records)} images to a VIMS spectrum.")

    with open(_RESOLVE_CACHE, "wb") as _f:
        _pkl.dump(records, _f)
    print(f"Cache written → {_RESOLVE_CACHE.name}")


In [ ]:
# ── Pre-load VIMS cubes for the 56 contrast images ───────────────────────────
# Identifies every flyby referenced by contrast_df, then loads all associated
# VIMS cubes into _cube_cache so the interactive viewer below starts instantly.

_contrast_flybys:  set[str] = set()
_contrast_cube_ids: list[str] = []

for _, _row in contrast_df.iterrows():
    _img = "N" + _row["opusid"].split("-n")[-1].upper()
    _fb  = imgid_to_flyby.get(_img)
    if _fb and _fb in flyby_to_cubeids:
        _contrast_flybys.add(_fb)

for _fb in sorted(_contrast_flybys):
    _contrast_cube_ids.extend(flyby_to_cubeids[_fb])

print(f"Contrast images → {len(_contrast_flybys)} flybys, {len(_contrast_cube_ids)} cubes to cache")

try:
    from tqdm.notebook import tqdm as _tqdm
    _iterable = _tqdm(_contrast_cube_ids, desc="Loading VIMS cubes")
except ImportError:
    _iterable = _contrast_cube_ids

for _cid in _iterable:
    get_cube(_cid)

_n_ok = sum(1 for c in _cube_cache.values() if c is not None)
print(f"Cache warm: {_n_ok}/{len(_contrast_cube_ids)} cubes loaded  "
      f"({len(_contrast_cube_ids) - _n_ok} unavailable)")


## Geometry plot: contrast subset with VIMS flyby labels

In [ ]:
# ── Geometry for the contrast-computed subset ──────────────────────────────────
# Only opusids present in 14_contrast_table.csv (those with a picks.json that
# were successfully processed by script 14).

from collections import defaultdict
from matplotlib.patches import Circle, Wedge
import matplotlib.patheffects as pe

_RS_KM = 60268.0
_SC    = "CASSINI"
_ENCELADUS = "ENCELADUS"

_MOON_ORBITS = {
    "F": 140220 / _RS_KM,
    "G": 167500 / _RS_KM,
    "E": 237948 / _RS_KM,
    "T": 294619 / _RS_KM,
    "D": 377396 / _RS_KM,
    "R": 527108 / _RS_KM,
}

# contrast_df and imgid_to_flyby are already built in cells above
geo_records = []
et_vals_geo = []

for _, row in contrast_df.iterrows():
    opusid   = row["opusid"]
    image_id = "N" + opusid.split("-n")[-1].upper()

    try:
        et = spice.scs2e(CASSINI_ID, "1/" + image_id[1:])
    except Exception as e:
        print(f"  {opusid}: SPICE error ({e}) — skipped")
        continue

    flyby = imgid_to_flyby.get(image_id, "?")
    et_vals_geo.append(et)
    geo_records.append({"opusid": opusid, "image_id": image_id, "et": et, "flyby": flyby})

# Reference ET: median across all contrast images
_et_ref = float(np.median(et_vals_geo))

# Saturn-frame rotation (IAU_SATURN) + rotate so Sun is at +y
_rot_ref = spice.pxform("J2000", "IAU_SATURN", _et_ref)
_pos_sun, _ = spice.spkpos("SUN", _et_ref, "J2000", "LT+S", "SATURN")
_sun_sat = _rot_ref.T @ np.array(_pos_sun)
_angle_rot = math.atan2(float(_sun_sat[1]), float(_sun_sat[0])) - math.pi / 2.0

def _rot2d(x, y):
    ca, sa = math.cos(-_angle_rot), math.sin(-_angle_rot)
    return x * ca - y * sa, x * sa + y * ca

def _to_rs(pos_j2000):
    p = _rot_ref.T @ np.asarray(pos_j2000, dtype=float)
    x, y = _rot2d(float(p[0]), float(p[1]))
    return x / _RS_KM, y / _RS_KM

# Sun direction arrow in 2-D plot
_sx = float(_sun_sat[0]); _sy = float(_sun_sat[1])
_sx2, _sy2 = _rot2d(_sx, _sy)
_sn = math.sqrt(_sx2**2 + _sy2**2)
_sun_dir_2d = np.array([_sx2, _sy2]) / _sn if _sn > 0 else np.array([0., 1.])

# Compute Cassini + Enceladus positions for each record
for rec in geo_records:
    try:
        pos_c, _   = spice.spkpos(_SC,        rec["et"], "J2000", "NONE", "SATURN")
        pos_e, _   = spice.spkpos(_ENCELADUS, rec["et"], "J2000", "NONE", "SATURN")
        rec["obs_x"], rec["obs_y"] = _to_rs(pos_c)
        rec["enc_x"], rec["enc_y"] = _to_rs(pos_e)
        rec["ok"] = True
    except Exception as e:
        rec["ok"] = False
        print(f"  {rec['opusid']}: geometry error ({e})")

_valid_geo = [r for r in geo_records if r.get("ok")]
print(f"Geometry computed for {len(_valid_geo)}/{len(geo_records)} contrast images")
print("Flyby codes found:", sorted({r['flyby'] for r in _valid_geo}))


In [ ]:
# ── Plot: geometry of contrast subset, labelled by VIMS flyby ─────────────────
# Sun is at the top (+y). Sub-solar ansa = top of E ring, anti-solar = bottom.
# The two lateral ansae (±x) are the morning and evening ansae; which is which
# depends on the sense of Enceladus' orbital motion (counter-clockwise when
# viewed from Saturn's north pole in this frame).

_r_max = max(
    max(math.hypot(r["obs_x"], r["obs_y"]) for r in _valid_geo),
    max(math.hypot(r["enc_x"], r["enc_y"]) for r in _valid_geo),
) * 1.08
_r_max = max(_r_max, 9.5)

fig, ax = plt.subplots(figsize=(12, 12), dpi=150)
ax.set_facecolor("#eaf4f7")

# Saturn disc
saturn_patch = Circle((0, 0), 1.0, facecolor="white", edgecolor="grey", lw=1.5, zorder=3)
ax.add_patch(saturn_patch)
shadow_patch = Wedge((0, 0), 1.0, 180, 360, facecolor="#cfcfcf", edgecolor="none", zorder=3)
ax.add_patch(shadow_patch)

# Ring orbits
for _lbl, _r in _MOON_ORBITS.items():
    _is_e = _lbl == "E"
    ax.add_patch(Circle((0, 0), _r, fill=False, edgecolor="grey",
                         lw=1.4 if _is_e else 1.0,
                         linestyle="-" if _is_e else "--",
                         alpha=0.9 if _is_e else 0.6, zorder=2))
    ax.text(_r + 0.15, 0.0, _lbl, fontsize=12, color="grey", va="center")

# Sun label
_sun_lbl_pos = _sun_dir_2d * (_MOON_ORBITS["R"] + 0.4)
ax.text(_sun_lbl_pos[0], _sun_lbl_pos[1], "Sun ☉",
        ha="center", va="center", fontsize=13, color="grey")

# Ansa / quadrant labels on E ring
_enc_r = _MOON_ORBITS["E"]
_ansa_kw = dict(fontsize=9, color="#555555", style="italic",
                ha="center", va="center", zorder=11,
                path_effects=[pe.withStroke(linewidth=2.5, foreground="#eaf4f7")])
ax.text(0,          _enc_r + 0.55,  "Sub-solar ansa",   **_ansa_kw)
ax.text(0,         -_enc_r - 0.55,  "Anti-solar ansa",  **_ansa_kw)
ax.text(-_enc_r - 0.65, 0,          "Morning ansa",      **_ansa_kw)
ax.text( _enc_r + 0.65, 0,          "Evening ansa",      **_ansa_kw)

# Sight-lines, Cassini and Enceladus dots
_COL_LINE  = "#607d8b"
_COL_CAS   = "#7b1fa2"
_COL_ENC   = "dodgerblue"

_flyby_enc: dict[str, list] = defaultdict(list)

for rec in _valid_geo:
    ax.plot([rec["obs_x"], rec["enc_x"]], [rec["obs_y"], rec["enc_y"]],
            color=_COL_LINE, lw=0.9, alpha=0.18, zorder=5)
    ax.scatter([rec["obs_x"]], [rec["obs_y"]],
               s=11, c=_COL_CAS, edgecolors="white", linewidths=0.2, alpha=0.55, zorder=7)
    ax.scatter([rec["enc_x"]], [rec["enc_y"]],
               s=15, c=_COL_ENC, edgecolors="white", linewidths=0.25, alpha=0.85, zorder=8)
    _flyby_enc[rec["flyby"]].append((rec["enc_x"], rec["enc_y"]))

from adjustText import adjust_text

_texts = []
# Label each flyby at the centroid of its Enceladus positions
for _flyby, _pts in sorted(_flyby_enc.items()):
    _mx = float(np.mean([p[0] for p in _pts]))
    _my = float(np.mean([p[1] for p in _pts]))
    _texts.append(ax.text(
        _mx, _my, _flyby,
        fontsize=7.5, color="#1a1a2e", zorder=13,
        path_effects=[pe.withStroke(linewidth=2.5, foreground="white")],
    ))

adjust_text(_texts, arrowprops=dict(arrowstyle="-", color="grey", lw=0.5, alpha=0.6))

# Legend
ax.scatter([], [], s=11, c=_COL_CAS, edgecolors="white", linewidths=0.5, label="Cassini")
ax.scatter([], [], s=15, c=_COL_ENC, edgecolors="white", linewidths=0.5, label="Enceladus (band position)")
ax.plot([], [], color=_COL_LINE, lw=1.0, alpha=0.35, label="Line of sight")
ax.legend(loc="upper right", fontsize=9, frameon=True)

ax.set_aspect("equal", adjustable="box")
ax.set_xlim(-_r_max, _r_max)
ax.set_ylim(-_r_max, _r_max)
ax.set_xlabel("Distance [Saturn radii]", fontsize=11)
ax.set_ylabel("Distance [Saturn radii]", fontsize=11)
ax.set_title(
    f"Viewing geometry — {len(_valid_geo)} contrast-computed observations\n"
    "labelled by VIMS flyby (Enceladus positions; Sun at top)",
    fontsize=12, pad=12,
)
ax.grid(False)
fig.tight_layout()
plt.show()


In [ ]:
# ── Second figure: all 624 pairs colour-coded by flyby, 45° ansa separators ──
# Positions come from 12_selected_bundle_A_geometry_overview.json (pre-computed
# by script 12 in the same Sun-at-top rotated Saturn frame as the plot above).
# The 23 ISS flyby sequences from obs_sequences.csv each get a distinct colour.

import json as _json
import matplotlib.cm as cm

_GEO_JSON_PATH = _STORAGE / "post_pipeline" / "12_selected_bundle_A_geometry_overview.json"
_geo_data   = _json.loads(_GEO_JSON_PATH.read_text())
_all_pairs  = _geo_data["all_pairs_records"]   # 624 records with enc_x, enc_y

# Attach flyby code to each record via the same imgid_to_flyby map
for _ap in _all_pairs:
    _ap["flyby"] = imgid_to_flyby.get(_ap["image_id"], "?")

# Ordered list of the 23 ISS flyby sequences (source of truth)
_FLYBY_ORDER2 = sorted(obs_df["obs_sequence"].dropna().unique())
_N_FB2 = len(_FLYBY_ORDER2)

# 23 maximally distinct colours: first 20 from tab20, next 3 from tab20b
_colors23 = [plt.cm.tab20(i) for i in range(20)] + [plt.cm.tab20b(i) for i in range(3)]
_flyby_col2 = {fb: _colors23[i] for i, fb in enumerate(_FLYBY_ORDER2)}
_flyby_col2["?"] = (0.55, 0.55, 0.55, 1.0)

# Compute r_max from the full dataset
_all_enc_r = [math.hypot(float(r["enc_x"]), float(r["enc_y"])) for r in _all_pairs]
_r_max2 = max(_all_enc_r) * 1.08
_r_max2 = max(_r_max2, 9.5)

fig2, ax2 = plt.subplots(figsize=(13, 13), dpi=150)
ax2.set_facecolor("#eaf4f7")

# Saturn disc
ax2.add_patch(Circle((0, 0), 1.0, facecolor="white", edgecolor="grey", lw=1.5, zorder=3))
ax2.add_patch(Wedge((0, 0), 1.0, 180, 360, facecolor="#cfcfcf", edgecolor="none", zorder=3))

# Ring orbits
for _lbl2, _r2 in _MOON_ORBITS.items():
    _is_e2 = _lbl2 == "E"
    ax2.add_patch(Circle((0, 0), _r2, fill=False, edgecolor="grey",
                          lw=1.4 if _is_e2 else 1.0,
                          linestyle="-" if _is_e2 else "--",
                          alpha=0.9 if _is_e2 else 0.6, zorder=2))
    ax2.text(_r2 + 0.15, 0.0, _lbl2, fontsize=12, color="grey", va="center")

# ── Ansa separator lines at ±45° (y = x and y = −x) ─────────────────────────
# The four ansae are centred on the cardinal directions (top/bottom/left/right).
# Their boundaries fall at the diagonals, not the axes.
_sep_kw = dict(color="#777777", lw=1.3, ls=(0, (5, 4)), alpha=0.85, zorder=4)
ax2.plot([-_r_max2, _r_max2], [-_r_max2,  _r_max2], **_sep_kw)   # y =  x
ax2.plot([-_r_max2, _r_max2], [ _r_max2, -_r_max2], **_sep_kw)   # y = -x

# Ansa quadrant labels — placed at the cardinal tips of the E ring
_enc_r2 = _MOON_ORBITS["E"]
_ql2    = _enc_r2 + 0.65
_ansa_kw2 = dict(fontsize=10, color="#333333", fontstyle="italic", fontweight="bold",
                 ha="center", va="center", zorder=12,
                 path_effects=[pe.withStroke(linewidth=3, foreground="#eaf4f7")])
ax2.text( 0,    _ql2,        "Sub-solar ansa",  **_ansa_kw2)
ax2.text( 0,   -_ql2,        "Anti-solar ansa", **_ansa_kw2)
ax2.text(-_ql2 - 0.3, 0,     "Morning ansa",    **_ansa_kw2)
ax2.text( _ql2 + 0.3, 0,     "Evening ansa",    **_ansa_kw2)

# Sun label (reuse direction computed in geo-vims-compute)
_sun_lbl2 = _sun_dir_2d * (_MOON_ORBITS["R"] + 0.4)
ax2.text(_sun_lbl2[0], _sun_lbl2[1], "Sun ☉",
         ha="center", va="center", fontsize=13, color="grey")

# All 624 Enceladus positions, coloured by flyby
for _ap in _all_pairs:
    _col2 = _flyby_col2.get(_ap["flyby"], _flyby_col2["?"])
    ax2.scatter([float(_ap["enc_x"])], [float(_ap["enc_y"])],
                s=18, color=_col2, edgecolors="white", linewidths=0.35,
                zorder=9, alpha=0.88)

_n_unknown = sum(1 for r in _all_pairs if r["flyby"] == "?")

from adjustText import adjust_text
import numpy as np

# Label the clusters instead of using a legend
texts = []
for _fb2 in _FLYBY_ORDER2 + (["?"] if _n_unknown else []):
    _fb_pts = [(float(_ap["enc_x"]), float(_ap["enc_y"])) for _ap in _all_pairs if _ap["flyby"] == _fb2]
    if _fb_pts:
        # Pick the middle point from the sequence to guarantee the label anchors to a real point
        # (Using the mean position of an arc pulls the anchor into empty space inside the orbit)
        _mid_idx = len(_fb_pts) // 2
        _mx, _my = _fb_pts[_mid_idx]
        
        _lbl = f"unassigned ({_n_unknown})" if _fb2 == "?" else _fb2
        texts.append(ax2.text(_mx, _my, _lbl, fontsize=8.5, color=_flyby_col2[_fb2], fontweight="bold"))

adjust_text(texts, ax=ax2, arrowprops=dict(arrowstyle="->", color="#555555", lw=0.8, alpha=0.8))

ax2.set_aspect("equal", adjustable="box")
ax2.set_xlim(-_r_max2, _r_max2)
ax2.set_ylim(-_r_max2, _r_max2)
ax2.set_xlabel("Distance [Saturn radii]", fontsize=11)
ax2.set_ylabel("Distance [Saturn radii]", fontsize=11)
ax2.set_title(
    f"Viewing geometry — {len(_all_pairs)} pairs across all {_N_FB2} flyby sequences\n"
    "Enceladus positions colour-coded by ISS flyby  "
    "(Sun at top; diagonal dotted lines separate ansae)",
    fontsize=12, pad=12,
)
ax2.grid(False)
fig2.tight_layout()
plt.show()


## Interactive flyby viewer — Rubbrecht-style spectral analysis

Cubes are displayed as a **false-colour composite** (R = 3.1 µm, G = 2.0 µm, B = 1.78 µm),
matching the VIMS portal preview — this makes the luminous band visible before you draw anything.

Use the **mode toggle**, then **left-click** on any cube:

| Mode | Colour | Interaction |
|---|---|---|
| Band | red | Each click adds a waypoint; pixels along the polyline are sampled |
| Background | blue | Click to place **■** ; click the same pixel again to remove it |
| Plume | green | Click to place **▲** ; click the same pixel again to remove it |

- **Right-click** undoes the last waypoint (Band) or last point (Background / Plume).
- **Clear all** resets every selection across all modes.
- **Aggregate & plot** produces the Rubbrecht-style scatter + median ± MAD figure.


In [ ]:
%matplotlib widget
import ipywidgets as widgets
from IPython.display import display

plt.ioff()

# Exactly the flybys present in obs_sequences.csv (the 23 ISS sequences),
# filtered to those that also have VIMS cubes.
_ISS_FLYBYS = set(obs_df["obs_sequence"].dropna().unique())
_FLYBYS = sorted(f for f in _ISS_FLYBYS if f in flyby_to_cubeids)

_MODES       = ["Band", "Background", "Plume"]
_MODE_COL    = {"Band": "#d62728", "Background": "#1f77b4", "Plume": "#2ca02c"}
_MODE_MARKER = {"Background": "s", "Plume": "^"}
_RGB_WVLNS   = (3.1, 2.0, 1.78)
_WVLN_PRESETS = [1.78, 2.00, 3.10, 3.45, 5.00]

# ── Persistent spectral store ─────────────────────────────────────────────────
_regions: dict[str, list] = {m: [] for m in _MODES}
_undo_stack: list = []

# ── Per-figure state (reset on Load) ─────────────────────────────────────────
_active_map:    list = []      # [(ax_rgb, ax_gray, cid, cube)]
_gray_ims:      list = []      # [(AxesImage, cube)] for in-place gray update
_band_wp:       dict = {}
_band_line_art: dict = {}
_band_dot_art:  dict = {}
_pt_entries:    list = []

# ── RGB composite helper ──────────────────────────────────────────────────────
def _make_rgb(c, wvlns=_RGB_WVLNS):
    try:
        rgb = np.asarray(c[wvlns[0], wvlns[1], wvlns[2]], dtype=np.float32)
    except Exception:
        channels = []
        for w in wvlns:
            try:    channels.append(np.asarray(c @ w, dtype=np.float32))
            except: channels.append(np.zeros((c.NL, c.NS), dtype=np.float32))
        rgb = np.stack(channels, axis=-1)
    for ch in range(3):
        plane  = rgb[:, :, ch]
        finite = plane[np.isfinite(plane)]
        if finite.size > 1:
            lo, hi = np.percentile(finite, 1), np.percentile(finite, 99)
            rgb[:, :, ch] = np.clip((plane - lo) / max(hi - lo, 1e-9), 0, 1)
    return np.nan_to_num(rgb, nan=0.0)

# ── Two persistent figures — created ONCE, reused forever ─────────────────────
_fig_cubes, _ = plt.subplots(1, 1, figsize=(10, 7))
_fig_cubes.clear()
_fig_cubes.canvas.header_visible = False

_fig_spec, _ax_spec = plt.subplots(figsize=(11, 3.5))
_fig_spec.canvas.header_visible = False

def _redraw_spec():
    _ax_spec.cla()
    for mode in _MODES:
        col = _MODE_COL[mode]
        for e in _regions[mode]:
            _ax_spec.plot(e["wvlns"], e["spec"], color=col, lw=0.8, alpha=0.45)
    for mode in _MODES:
        if _regions[mode]:
            _ax_spec.plot([], [], color=_MODE_COL[mode], lw=2, label=mode)
    if any(_regions[m] for m in _MODES):
        _ax_spec.legend(fontsize=8)
    _ax_spec.set_xlabel("Wavelength (µm)")
    _ax_spec.set_ylabel("I/F")
    _ax_spec.set_title("Selected pixel spectra")
    _fig_spec.canvas.draw_idle()

_redraw_spec()

# ── Band polyline ─────────────────────────────────────────────────────────────
def _pixels_along_polyline(waypoints, NS, NL):
    seen, out = set(), []
    for i in range(len(waypoints) - 1):
        s0, l0 = waypoints[i]; s1, l1 = waypoints[i + 1]
        n = max(abs(s1 - s0), abs(l1 - l0)) + 1
        for s, l in zip(np.round(np.linspace(s0, s1, n)).astype(int),
                        np.round(np.linspace(l0, l1, n)).astype(int)):
            k = (int(s), int(l))
            if k not in seen and 1 <= s <= NS and 1 <= l <= NL:
                seen.add(k); out.append(k)
    return out

def _extract_band_spectrum_interp(c, waypoints):
    """Per-wavelength cubic spline interpolation along the band polyline.
    Returns one averaged spectrum for the whole line (Dhingra et al. 2017)."""
    from scipy.ndimage import map_coordinates
    all_r, all_c = [], []
    for i in range(len(waypoints) - 1):
        s0, l0 = waypoints[i]; s1, l1 = waypoints[i + 1]
        # 3× oversampling for sub-pixel accuracy
        n = max(int(np.hypot(s1 - s0, l1 - l0) * 3) + 1, 2)
        endpoint = (i == len(waypoints) - 2)
        all_r.extend(np.linspace(l0 - 1, l1 - 1, n, endpoint=endpoint))  # row = line - 1
        all_c.extend(np.linspace(s0 - 1, s1 - 1, n, endpoint=endpoint))  # col = sample - 1
    coords = np.array([all_r, all_c])  # shape (2, N)
    wvlns = np.asarray(c.wvlns)
    spectrum = np.full(len(wvlns), np.nan)
    for wi, wvln in enumerate(wvlns):
        try:
            img = np.asarray(c @ wvln, dtype=np.float64)
            vals = map_coordinates(img, coords, order=3, mode='nearest')
            finite = vals[np.isfinite(vals)]
            if finite.size:
                spectrum[wi] = finite.mean()
        except Exception:
            pass
    return spectrum, wvlns

def _update_band_line(ax_id, ax, c, cid):
    for d in (_band_line_art, _band_dot_art):
        art = d.pop(ax_id, None)
        if art:
            try: art.remove()
            except: pass
    wp = _band_wp.get(ax_id, [])
    if not wp:
        _fig_cubes.canvas.draw_idle(); return
    line, = ax.plot([p[0] for p in wp], [p[1] for p in wp],
                    color=_MODE_COL["Band"], lw=2.0,
                    marker="o", markersize=4, markerfacecolor="white", zorder=11)
    _band_line_art[ax_id] = line
    if len(wp) >= 2:
        pixels = _pixels_along_polyline(wp, c.NS, c.NL)
        if pixels:
            ps = np.array(pixels)
            dot = ax.scatter(ps[:, 0], ps[:, 1], s=12,
                             color=_MODE_COL["Band"], alpha=0.35,
                             linewidths=0, zorder=10)
            _band_dot_art[ax_id] = dot
            _regions["Band"] = [e for e in _regions["Band"]
                                 if e.get("_ax_id") != ax_id]
            spec, wvlns = _extract_band_spectrum_interp(c, wp)
            if np.any(np.isfinite(spec)):
                _regions["Band"].append({
                    "label": f"{cid} (band)", "_ax_id": ax_id,
                    "spec": spec, "wvlns": wvlns,
                })
    _fig_cubes.canvas.draw_idle()
    _redraw_spec()

# ── Refresh gray in-place ─────────────────────────────────────────────────────
def _refresh_gray(wvln: float):
    for im, c in _gray_ims:
        try:
            data   = c @ wvln
            finite = data[np.isfinite(data)]
            im.set_data(data)
            im.set_clim(0, float(np.percentile(finite, 99)) if finite.size else 1.0)
        except: pass
    _fig_cubes.canvas.draw_idle()

# ── Build figure — clears selection and redraws the SAME canvas ───────────────
def _build_figure(cube_id: str):
    # Clear selection every time a new cube is loaded
    for m in _MODES: _regions[m].clear()
    _undo_stack.clear()
    _band_wp.clear(); _band_line_art.clear(); _band_dot_art.clear()
    _pt_entries.clear(); _active_map.clear(); _gray_ims.clear()
    _redraw_spec()

    c = get_cube(cube_id)
    loaded = [(cube_id, c)] if c is not None else []

    _fig_cubes.clear()

    if not loaded:
        _fig_cubes.text(0.5, 0.5, f"No cubes available for {cube_id}",
                        ha="center", va="center", fontsize=12)
        _fig_cubes.canvas.draw_idle()
        return

    n     = len(loaded)
    ncols = min(n, 3)
    nrows = (n + ncols - 1) // ncols

    _fig_cubes.set_size_inches(5.2 * ncols, 7.0 * nrows)
    axs = _fig_cubes.subplots(nrows * 2, ncols, squeeze=False)

    init_wvln = _wvln_custom.value

    for idx, (cid, c) in enumerate(loaded):
        cube_row, col = divmod(idx, ncols)
        ax_rgb  = axs[cube_row * 2,     col]
        ax_gray = axs[cube_row * 2 + 1, col]
        _active_map.append((ax_rgb, ax_gray, cid, c))

        # RGB row
        try:
            ax_rgb.imshow(_make_rgb(c), extent=c.extent, aspect="auto", origin="upper")
        except Exception as e:
            ax_rgb.text(0.5, 0.5, f"RGB error\n{e}", transform=ax_rgb.transAxes,
                        ha="center", va="center", fontsize=7)
        ax_rgb.set_title(f"RGB  |  {cid}", fontsize=7, pad=2)
        ax_rgb.set_xlabel(c.slabel, fontsize=6); ax_rgb.set_ylabel(c.llabel, fontsize=6)
        ax_rgb.tick_params(labelsize=6)
        try: ax_rgb.set_xticks(c.sticks); ax_rgb.set_yticks(c.lticks)
        except: pass

        # Gray row
        try:
            data   = c @ init_wvln
            finite = data[np.isfinite(data)]
            im_g   = ax_gray.imshow(data, extent=c.extent, cmap="gray",
                                     vmin=0,
                                     vmax=float(np.percentile(finite, 99))
                                          if finite.size else 1.0,
                                     aspect="auto", origin="upper")
            _gray_ims.append((im_g, c))
        except Exception as e:
            ax_gray.text(0.5, 0.5, f"Error\n{e}", transform=ax_gray.transAxes,
                         ha="center", va="center", fontsize=7)
        ax_gray.set_title(f"Gray  {init_wvln:.2f} µm", fontsize=7, pad=2)
        ax_gray.set_xlabel(c.slabel, fontsize=6); ax_gray.set_ylabel(c.llabel, fontsize=6)
        ax_gray.tick_params(labelsize=6)
        try: ax_gray.set_xticks(c.sticks); ax_gray.set_yticks(c.lticks)
        except: pass

    for idx in range(len(loaded), nrows * ncols):
        cube_row, col = divmod(idx, ncols)
        axs[cube_row * 2,     col].set_visible(False)
        axs[cube_row * 2 + 1, col].set_visible(False)

    def _on_click(event):
        if event.inaxes is None: return
        for ax_rgb, ax_gray, cid, c in _active_map:
            if event.inaxes not in (ax_rgb, ax_gray): continue
            ax_id = id(ax_rgb)
            mode  = _mode_toggle.value
            s = int(round(event.xdata))
            l = int(round(event.ydata))

            if event.button == 3:      # right-click: undo
                if _undo_stack:
                    u_mode, u_type, u_ax_id = _undo_stack.pop()
                    if u_type == "waypoint":
                        wp = _band_wp.get(u_ax_id, [])
                        if wp: wp.pop()
                        _update_band_line(u_ax_id, ax_rgb, c, cid)
                    else:
                        for i in range(len(_pt_entries) - 1, -1, -1):
                            e = _pt_entries[i]
                            if e["mode"] == u_mode:
                                try: e["art"].remove(); e["art_g"].remove()
                                except: pass
                                _pt_entries.pop(i)
                                if _regions[u_mode]: _regions[u_mode].pop()
                                break
                        _fig_cubes.canvas.draw_idle(); _redraw_spec()
                return

            if not (1 <= s <= c.NS and 1 <= l <= c.NL): return

            if mode == "Band":
                wp = _band_wp.setdefault(ax_id, [])
                if (s, l) in wp:          # click existing waypoint → remove it
                    wp.remove((s, l))
                    for i in range(len(_undo_stack) - 1, -1, -1):
                        if _undo_stack[i] == ("Band", "waypoint", ax_id):
                            _undo_stack.pop(i); break
                else:                     # new waypoint
                    wp.append((s, l))
                    _undo_stack.append(("Band", "waypoint", ax_id))
                _update_band_line(ax_id, ax_rgb, c, cid)
                _fig_cubes.canvas.draw_idle()
                return

            # Toggle exact pixel
            for i, e in enumerate(_pt_entries):
                if e["mode"] == mode and e["ax_id"] == ax_id \
                        and e["s"] == s and e["l"] == l:
                    try: e["art"].remove(); e["art_g"].remove()
                    except: pass
                    _pt_entries.pop(i)
                    lbl = f"{cid} ({s},{l})"
                    _regions[mode] = [r for r in _regions[mode]
                                      if r.get("label") != lbl]
                    _undo_stack[:] = [u for u in _undo_stack
                                      if not (u[0] == mode and u[1] == "point"
                                              and u[2] == ax_id)]
                    _fig_cubes.canvas.draw_idle(); _redraw_spec()
                    return

            try:
                spec  = c[s, l].spectrum.copy()
                wvlns = c.wvlns.copy()
            except: return
            art   = ax_rgb.scatter([s], [l], s=80, color=_MODE_COL[mode],
                                    marker=_MODE_MARKER[mode],
                                    edgecolors="white", linewidths=1.2, zorder=11)
            art_g = ax_gray.scatter([s], [l], s=80, color=_MODE_COL[mode],
                                     marker=_MODE_MARKER[mode],
                                     edgecolors="white", linewidths=1.2, zorder=11)
            lbl = f"{cid} ({s},{l})"
            _pt_entries.append({"art": art, "art_g": art_g, "mode": mode,
                                  "ax_id": ax_id, "s": s, "l": l})
            _regions[mode].append({"label": lbl, "spec": spec, "wvlns": wvlns})
            _undo_stack.append((mode, "point", ax_id))
            _fig_cubes.canvas.draw_idle(); _redraw_spec()
            return

    # Disconnect any previous handler before connecting the new one,
    # otherwise every Load adds a new handler and clicks fire twice.
    if hasattr(_fig_cubes, '_click_cid') and _fig_cubes._click_cid is not None:
        _fig_cubes.canvas.mpl_disconnect(_fig_cubes._click_cid)
    _fig_cubes._click_cid = _fig_cubes.canvas.mpl_connect("button_press_event", _on_click)
    _fig_cubes.tight_layout(h_pad=0.5)
    _fig_cubes.canvas.draw_idle()

# ── Ansa helper (shared by save PNG and save JSON) ────────────────────────────
def _get_ansa(fb: str) -> str:
    fb_pts = [(float(_ap["enc_x"]), float(_ap["enc_y"])) for _ap in _all_pairs if _ap["flyby"] == fb]
    if not fb_pts:
        return "unknown"
    mx = np.mean([p[0] for p in fb_pts])
    my = np.mean([p[1] for p in fb_pts])
    if   my >  abs(mx): return "subsolar"
    elif my < -abs(mx): return "antisolar"
    elif mx >  abs(my): return "evening"
    else:               return "morning"

# ── Aggregate plot ────────────────────────────────────────────────────────────
def _aggregate_plot(_=None, save=False):
    if not any(_regions[m] for m in _MODES):
        print("No pixels selected yet."); return
    fig_agg, ax_agg = plt.subplots(figsize=(10, 8))
    fig_agg.canvas.header_visible = False
    for mode in _MODES:
        entries = _regions[mode]
        if not entries: continue
        col       = _MODE_COL[mode]
        ref_wvlns = entries[0]["wvlns"]
        spectra   = [e["spec"] for e in entries
                     if len(e["wvlns"]) == len(ref_wvlns)]
        if not spectra: continue
        arr = np.array(spectra)
        med = np.nanmedian(arr, axis=0)
        ax_agg.scatter(ref_wvlns, med, color=col, s=18, zorder=6,
                       label=f"{mode}  (n={arr.shape[0]})")
    ax_agg.axvline(2.85, color="k", lw=1.0, ls="--", alpha=0.6, label="2.85 µm")
    ax_agg.set_ylim(bottom=0)
    ax_agg.set_xlabel("λ [µm]", fontsize=11); ax_agg.set_ylabel("I/F", fontsize=11)
    ax_agg.set_title("VIMS spectra by region — median", fontsize=10)
    ax_agg.grid(True, lw=0.4, alpha=0.5)
    ax_agg.legend(fontsize=9); fig_agg.tight_layout()

    if save:
        fb   = _dd.value
        ansa = _get_ansa(fb)
        out_dir = Path("/home/arnaudm/projects/LuminousBandsInERing/mnt-storage/post_pipeline/17_VIMS_campain")
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{ansa}_{fb}_{_dd_cube.value}.png"
        fig_agg.savefig(out_path, dpi=200, bbox_inches="tight")
        plt.close(fig_agg)
        print(f"Saved plot: {out_path.name}")
    else:
        display(fig_agg.canvas)

def _on_save(_):
    _aggregate_plot(save=True)

# ── Save JSON ────────────────────────────────────────────────────────────────
def _on_save_json(_):
    if not any(_regions[m] for m in _MODES):
        print("No pixels selected yet."); return
    fb   = _dd.value
    ansa = _get_ansa(fb)
    payload = {
        "flyby": fb,
        "ansa":  ansa,
        "regions": {
            mode: [
                {
                    "label": e["label"],
                    "wvlns": np.asarray(e["wvlns"]).tolist(),
                    "spec":  np.where(np.isfinite(e["spec"]), e["spec"], None).tolist(),
                }
                for e in _regions[mode]
            ]
            for mode in _MODES
        },
    }
    out_dir = Path("/home/arnaudm/projects/LuminousBandsInERing/mnt-storage/post_pipeline/17_VIMS_campain")
    out_dir.mkdir(parents=True, exist_ok=True)
    out_path = out_dir / f"{ansa}_{fb}_{_dd_cube.value}.json"
    out_path.write_text(json.dumps(payload, indent=2))
    print(f"Saved JSON: {out_path.name}  "
          f"({sum(len(payload['regions'][m]) for m in _MODES)} spectra)")

# ── Controls ──────────────────────────────────────────────────────────────────
_dd = widgets.Dropdown(options=_FLYBYS, description="Flyby:",
                        layout=widgets.Layout(width="210px"))
_dd_cube = widgets.Dropdown(options=[], description="Cube:",
                             layout=widgets.Layout(width="300px"))

def _update_cube_dd(change):
    cubes = flyby_to_cubeids.get(change["new"], [])
    _dd_cube.options = cubes
    _dd_cube.value   = cubes[0] if cubes else None

_dd.observe(_update_cube_dd, names="value")
if _FLYBYS:
    _update_cube_dd({"new": _FLYBYS[0]})

_btn_load = widgets.Button(description="Load cube", button_style="primary",
                            icon="refresh", layout=widgets.Layout(width="120px"))
_btn_next = widgets.Button(description="Next cube", button_style="primary",
                            icon="arrow-right", layout=widgets.Layout(width="120px"))

_wvln_dd = widgets.Dropdown(
    options=[(f"{w:.2f} µm", w) for w in _WVLN_PRESETS],
    value=2.00, description="Gray λ:",
    layout=widgets.Layout(width="170px"),
)
_wvln_custom = widgets.BoundedFloatText(
    value=2.00, min=0.88, max=5.12, step=0.01,
    description="Custom:", layout=widgets.Layout(width="170px"),
)
# Prevent circular updates between the two wavelength widgets
_wvln_syncing = [False]
def _on_wvln_dd(change):
    if _wvln_syncing[0]: return
    _wvln_syncing[0] = True
    _wvln_custom.value = change["new"]
    _wvln_syncing[0] = False
    _refresh_gray(change["new"])
def _on_wvln_custom(change):
    if _wvln_syncing[0]: return
    _refresh_gray(change["new"])
_wvln_dd.observe(_on_wvln_dd, names="value")
_wvln_custom.observe(_on_wvln_custom, names="value")

_mode_toggle = widgets.ToggleButtons(
    options=_MODES, value="Band",
    style={"button_width": "110px", "description_width": "0px"},
    tooltips=["Draw band polyline", "Add/remove background ■", "Add/remove plume ▲"],
)
_btn_clear     = widgets.Button(description="Clear all",        button_style="warning",
                                 layout=widgets.Layout(width="100px"))
_btn_agg       = widgets.Button(description="Aggregate & plot", button_style="success",
                                 layout=widgets.Layout(width="150px"))
_btn_save      = widgets.Button(description="Save plot",        button_style="info",
                                 layout=widgets.Layout(width="100px"))
_btn_save_json = widgets.Button(description="Save JSON",        button_style="",
                                 icon="download",               layout=widgets.Layout(width="110px"))

def _on_clear(_):
    for m in _MODES: _regions[m].clear()
    _undo_stack.clear()
    _redraw_spec()
    _build_figure(_dd_cube.value)

def _on_next(_):
    opts = list(_dd_cube.options)
    if not opts: return
    idx = opts.index(_dd_cube.value) if _dd_cube.value in opts else -1
    next_idx = min(idx + 1, len(opts) - 1)
    _dd_cube.value = opts[next_idx]
    _build_figure(_dd_cube.value)

_btn_load.on_click(lambda _: _build_figure(_dd_cube.value))
_btn_next.on_click(_on_next)
_btn_clear.on_click(_on_clear)
_btn_agg.on_click(_aggregate_plot)
_btn_save.on_click(_on_save)
_btn_save_json.on_click(_on_save_json)

# Display controls first, then figures separately.
# VS Code's Jupyter renderer cannot embed ipympl canvases inside a VBox;
# displaying them as independent outputs works around this.
display(widgets.VBox([
    widgets.HBox([_dd, _dd_cube, _btn_load, _btn_next, widgets.Label("   "), _wvln_dd, _wvln_custom]),
    widgets.HBox([widgets.Label("Mode:"), _mode_toggle, _btn_clear, _btn_agg, _btn_save, _btn_save_json]),
]))
display(_fig_cubes.canvas)
display(_fig_spec.canvas)